# AI Lab 16 - LightGBM Credit Card Fraud Detection on Kaggle Notebooks

Bài lab này thực hiện:
1. Load dataset Credit Card Fraud Detection (284,807 transactions)
2. Train LightGBMClassifier để phát hiện gian lận
3. Đo các metric: AUC-ROC, Accuracy, F1-Score, Precision, Recall
4. Đo inference latency và throughput
5. Lưu kết quả ra file `benchmark_result.json`

In [ ]:
import time
import json
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

print('LightGBM version:', lgb.__version__)
print('sklearn version:', __import__('sklearn').__version__)
print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

In [ ]:
# Load dataset
DATA_PATHS = [
    '/kaggle/input/creditcard-ulb/creditcard.csv',
    '/kaggle/input/creditcardfraud/creditcard.csv',
    '/kaggle/input/credit-card-fraud-detection/creditcard.csv',
]
data_path = None
for p in DATA_PATHS:
    if os.path.exists(p):
        data_path = p
        break
if data_path is None:
    # fallback: find any csv in /kaggle/input
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv'):
                data_path = os.path.join(root, f)
                break
        if data_path:
            break
print('Data path:', data_path)

t0 = time.time()
df = pd.read_csv(data_path)
load_time = time.time() - t0
print(f'Loaded {len(df):,} rows in {load_time:.2f}s')
print('Columns:', list(df.columns))
print('Class distribution:')
print(df['Class'].value_counts())

In [ ]:
# Preprocess
X = df.drop(columns=['Class'])
y = df['Class'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Train fraud ratio:', y_train.mean())
print('Test fraud ratio:', y_test.mean())

In [ ]:
# Train LightGBM
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'is_unbalance': True,
}

t0 = time.time()
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
train_time = time.time() - t0
best_iter = model.best_iteration
print(f'Training time: {train_time:.2f}s | Best iteration: {best_iter}')

In [ ]:
# Evaluate
y_pred_proba = model.predict(X_test, num_iteration=best_iter)
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'AUC-ROC:   {auc:.6f}')
print(f'Accuracy:  {acc:.6f}')
print(f'F1-Score:  {f1:.6f}')
print(f'Precision: {prec:.6f}')
print(f'Recall:    {rec:.6f}')
print('Confusion Matrix:')
print(cm)

In [ ]:
# Inference latency (1 row)
single_row = X_test.iloc[[0]]
t0 = time.time()
_ = model.predict(single_row, num_iteration=best_iter)
latency_1 = (time.time() - t0) * 1000.0  # ms
print(f'Latency (1 row): {latency_1:.4f} ms')

# Inference throughput (1000 rows, repeat 10x)
sample_1000 = X_test.iloc[:1000].copy()
t0 = time.time()
for _ in range(10):
    _ = model.predict(sample_1000, num_iteration=best_iter)
total_t = time.time() - t0
throughput = 1000 * 10 / total_t
print(f'Throughput (1000 rows, 10x): {throughput:.2f} rows/sec')

In [ ]:
# Save results
results = {
    'platform': 'Kaggle Notebooks (CPU)',
    'dataset': 'mlg-ulb/creditcardfraud',
    'rows_total': int(len(df)),
    'rows_train': int(len(X_train)),
    'rows_test': int(len(X_test)),
    'features': int(X.shape[1]),
    'load_time_sec': round(load_time, 4),
    'train_time_sec': round(train_time, 4),
    'best_iteration': int(best_iter),
    'auc_roc': round(float(auc), 6),
    'accuracy': round(float(acc), 6),
    'f1_score': round(float(f1), 6),
    'precision': round(float(prec), 6),
    'recall': round(float(rec), 6),
    'confusion_matrix': cm.tolist(),
    'inference_latency_ms_1row': round(latency_1, 4),
    'inference_throughput_rows_per_sec': round(throughput, 2),
}

out_path = '/kaggle/working/benchmark_result.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print('Saved:', out_path)
print(json.dumps(results, indent=2))